In [ ]:
import pyodbc
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

cnxn = pyodbc.connect('DSN=Hermes_DSN', autocommit=True)

In [ ]:
bond = pd.read_csv('C:\\Users\\hermesf\\Projects\\BanksFC\\bond_ts.csv')

In [ ]:
securities = tuple(bond['ISIN'].unique())

In [ ]:
# Data prep
query = f""" 

SELECT business_date, lender_id as entity_id, security_isin, sum(nominal_euro) as lending_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND intragroup = 0
AND s_lender.sector = 'BANK'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, lender_id, security_isin
ORDER BY business_date, lender_id, security_isin
""" 

df_lend = pd.read_sql_query(query, cnxn)

In [ ]:
# Data prep
query = f""" 

SELECT business_date, borrower_id as entity_id, security_isin, sum(nominal_euro) as borrowing_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND nominal_ccy IN ('EUR') 
AND intragroup = 0
AND s_borrower.sector = 'BANK'
AND gnlcoll = 'SPEC'
AND security_isin IN {securities}
GROUP BY business_date, borrower_id, security_isin
ORDER BY business_date, borrower_id, security_isin
""" 

df_borr = pd.read_sql_query(query, cnxn)

In [ ]:
df_lend['business_date'] = pd.to_datetime(df_lend['business_date'])
df_borr['business_date'] = pd.to_datetime(df_borr['business_date'])

In [ ]:
df = df_lend.merge(df_borr, on = ['business_date', 'entity_id', 'security_isin'], how = 'outer')

In [ ]:
# USD exposure: bank-level gross repo volume by cash currency
query = f""" 

SELECT business_date, lender_id as entity_id, nominal_ccy, sum(nominal_euro) as lending_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_lender ON s.lender_id = s_lender.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND intragroup = 0
AND s_lender.sector = 'BANK'
GROUP BY business_date, lender_id, nominal_ccy
ORDER BY business_date, lender_id, nominal_ccy
""" 

fx_lend = pd.read_sql_query(query, cnxn)

In [ ]:
query = f""" 

SELECT business_date, borrower_id as entity_id, nominal_ccy, sum(nominal_euro) as borrowing_volume

FROM xlab_ecb_prj_sftds_cb_common.hermesf_state s 
LEFT JOIN lab_prj_emir_ecb.mbf_sector_enrichment_20210531 s_borrower ON s.borrower_id = s_borrower.id
WHERE s.business_date >= '2021-01-04' AND s.business_date <= '2025-11-01' 
AND intragroup = 0
AND s_borrower.sector = 'BANK'
GROUP BY business_date, borrower_id, nominal_ccy
ORDER BY business_date, borrower_id, nominal_ccy
""" 

fx_borr = pd.read_sql_query(query, cnxn)

In [ ]:
# Bank-month USD share of total repo book (gross = lending + borrowing)
fx = fx_lend.merge(fx_borr, on = ['business_date', 'entity_id', 'nominal_ccy'], how = 'outer')
fx['gross'] = fx['lending_volume'].fillna(0) + fx['borrowing_volume'].fillna(0)
fx['month'] = pd.to_datetime(fx['business_date']).dt.to_period('M')

total = fx.groupby(['entity_id', 'month'])['gross'].sum()
usd = fx[fx['nominal_ccy'] == 'USD'].groupby(['entity_id', 'month'])['gross'].sum()
usd_share = (usd.reindex(total.index, fill_value = 0) / total).rename('usd_share').reset_index()

In [ ]:
# Bond-day exposure: trailing-30d EUR repo volume in the bond, weighted by
# banks' previous-month USD share (window ends at t-1 -> predetermined)
df['gross_eur'] = df['lending_volume'].fillna(0) + df['borrowing_volume'].fillna(0)
df['month'] = df['business_date'].dt.to_period('M')

d = df.merge(usd_share.assign(month = usd_share['month'] + 1), on = ['entity_id', 'month'])
d['wx'] = d['gross_eur'] * d['usd_share']

daily = d.groupby(['security_isin', 'business_date'], as_index = False)[['gross_eur', 'wx']].sum()
daily = daily.sort_values('business_date').set_index('business_date')
r = daily.groupby('security_isin')[['gross_eur', 'wx']].rolling('30D', closed = 'left').sum()

bond_exp = (r['wx'] / r['gross_eur']).rename('usd_exposure').reset_index().dropna()
bond_exp = bond_exp.rename(columns = {'security_isin': 'ISIN', 'business_date': 'date'})
bond_exp.to_csv('C:\\Users\\hermesf\\Projects\\BanksFC\\bond_usd_exposure.csv', index = False)